# 使用 Twitter API v2 获取最新技术资讯

本笔记本演示如何使用 Twitter API v2 搜索和获取感兴趣领域的最新技术推文。

**参考文档**: 
- [Twitter API v2 文档](https://developer.twitter.com/en/docs/twitter-api)
- [Recent Search API](https://developer.twitter.com/en/docs/twitter-api/tweets/search/api-reference/get-tweets-search-recent)

**特点**:
- ✅ 支持复杂搜索查询（关键词、时间范围、语言等）
- ✅ 实时获取最近7天的推文
- ✅ 获取作者信息、媒体附件等扩展数据
- ✅ 支持排序（相关性/时间）

## 步骤1: 安装必要的库

In [1]:
# 安装 tweepy (Python Twitter API 库)
!pip install tweepy python-dotenv -q
print("✓ 依赖库安装完成")

✓ 依赖库安装完成


## 步骤2: 导入必要的库

In [2]:
import json
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
import tweepy
from typing import List, Dict, Any, Optional

print("✓ 库导入成功")

✓ 库导入成功


## 步骤3: 加载环境变量和配置

从 `.env` 文件中加载 Twitter Bearer Token

In [3]:
# 加载环境变量
load_dotenv()

# 获取 Twitter API Bearer Token
TWITTER_BEARER_TOKEN = os.getenv('TWITTER_BEARER_TOKEN')

# 验证 Token 是否加载成功
if TWITTER_BEARER_TOKEN:
    print(f"✓ Twitter Bearer Token 已加载 (长度: {len(TWITTER_BEARER_TOKEN)} 字符)")
else:
    print("✗ 警告: Twitter Bearer Token 未找到，请检查.env文件")
    print("  请在.env文件中添加: TWITTER_BEARER_TOKEN=your_bearer_token_here")

✓ Twitter Bearer Token 已加载 (长度: 114 字符)


## 步骤4: 定义感兴趣的技术领域

自定义你想要获取资讯的技术领域和时间范围

In [4]:
# 定义感兴趣的技术领域（英文关键词）
TECH_FIELDS = [
    "autonomous driving",
    "embodied intelligence",
    "large language model",
    "artificial intelligence"
]

# 定义重点关注的主题
FOCUS_TOPICS = [
    "Tesla FSD",
    "humanoid robot",
    "OpenAI",
    "DeepSeek"
]

# 搜索配置
SEARCH_DAYS_BACK = 7  # 搜索最近几天的推文 (Twitter API 限制最多7天)
MAX_RESULTS = 100  # 每次请求返回的最大结果数 (10-100)
SORT_ORDER = "relevancy"  # 排序方式: "relevancy" 或 "recency"
LANGUAGE = "en"  # 语言过滤
EXCLUDE_RETWEETS = True  # 是否排除转推
EXCLUDE_REPLIES = True  # 是否排除回复

print(f"关注领域: {', '.join(TECH_FIELDS)}")
print(f"重点主题: {', '.join(FOCUS_TOPICS)}")
print(f"搜索时间范围: 最近 {SEARCH_DAYS_BACK} 天")
print(f"最大结果数: {MAX_RESULTS}")
print(f"排序方式: {SORT_ORDER}")
print(f"语言: {LANGUAGE}")
print(f"排除转推: {EXCLUDE_RETWEETS}")
print(f"排除回复: {EXCLUDE_REPLIES}")

关注领域: autonomous driving, embodied intelligence, large language model, artificial intelligence
重点主题: Tesla FSD, humanoid robot, OpenAI, DeepSeek
搜索时间范围: 最近 7 天
最大结果数: 100
排序方式: relevancy
语言: en
排除转推: True
排除回复: True


## 步骤5: 创建 Twitter API 客户端

封装 Twitter API v2 搜索功能

In [5]:
class TwitterSearchClient:
    """
    Twitter API v2 搜索客户端
    """
    
    def __init__(self, bearer_token: str):
        self.client = tweepy.Client(bearer_token=bearer_token)
        
    def build_query(self,
                   keywords: List[str],
                   language: str = "en",
                   exclude_retweets: bool = True,
                   exclude_replies: bool = True) -> str:
        """
        构建搜索查询字符串
        
        参数:
            keywords: 关键词列表
            language: 语言代码 (如 'en', 'zh')
            exclude_retweets: 是否排除转推
            exclude_replies: 是否排除回复
        
        返回:
            查询字符串
        """
        # 使用 OR 连接关键词，并用引号包裹包含空格的关键词
        keyword_parts = []
        for kw in keywords:
            if ' ' in kw:
                keyword_parts.append(f'("{kw}")')
            else:
                keyword_parts.append(kw)
        
        query_parts = [f"({' OR '.join(keyword_parts)})"]
        
        # 添加语言过滤
        if language:
            query_parts.append(f"lang:{language}")
        
        # 排除转推
        if exclude_retweets:
            query_parts.append("-is:retweet")
        
        # 排除回复
        if exclude_replies:
            query_parts.append("-is:reply")
        
        return " ".join(query_parts)
    
    def search_recent_tweets(self,
                            query: str,
                            start_time: datetime,
                            max_results: int = 100,
                            sort_order: str = "relevancy") -> Optional[Dict[str, Any]]:
        """
        搜索最近的推文
        
        参数:
            query: 搜索查询字符串
            start_time: 开始时间 (datetime 对象)
            max_results: 最大结果数 (10-100)
            sort_order: 排序方式 ('relevancy' 或 'recency')
        
        返回:
            搜索结果字典
        """
        try:
            # 调用 Twitter API v2 Recent Search
            response = self.client.search_recent_tweets(
                query=query,
                start_time=start_time,
                max_results=max_results,
                sort_order=sort_order,
                tweet_fields=['id', 'text', 'author_id', 'created_at', 'public_metrics', 
                             'entities', 'possibly_sensitive', 'lang'],
                expansions=['author_id', 'attachments.media_keys', 'geo.place_id'],
                media_fields=['url', 'preview_image_url', 'type'],
                user_fields=['name', 'username', 'verified', 'description', 'public_metrics'],
                place_fields=['country', 'geo', 'name']
            )
            
            # 转换为字典格式
            result = {
                'data': [],
                'includes': {},
                'meta': {}
            }
            
            # 处理推文数据
            if response.data:
                result['data'] = [tweet.data for tweet in response.data]
                result['meta'] = response.meta
            
            # 处理扩展数据 (用户、媒体等)
            if response.includes:
                if 'users' in response.includes:
                    result['includes']['users'] = [user.data for user in response.includes['users']]
                if 'media' in response.includes:
                    result['includes']['media'] = [media.data for media in response.includes['media']]
                if 'places' in response.includes:
                    result['includes']['places'] = [place.data for place in response.includes['places']]
            
            return result
            
        except tweepy.TweepyException as e:
            print(f"✗ Twitter API 调用失败: {e}")
            return None
        except Exception as e:
            print(f"✗ 搜索错误: {e}")
            return None

print("✓ TwitterSearchClient 类已定义")

✓ TwitterSearchClient 类已定义


## 步骤6: 执行推文搜索

使用 Twitter API 搜索最新技术相关推文

In [6]:
# 初始化客户端
if TWITTER_BEARER_TOKEN:
    client = TwitterSearchClient(TWITTER_BEARER_TOKEN)
    print("✓ Twitter API 客户端初始化成功")
    
    # 构建搜索查询
    all_keywords = TECH_FIELDS + FOCUS_TOPICS
    query = client.build_query(
        keywords=all_keywords,
        language=LANGUAGE,
        exclude_retweets=EXCLUDE_RETWEETS,
        exclude_replies=EXCLUDE_REPLIES
    )
    
    # 计算开始时间 (UTC时间)
    start_time = datetime.now(timezone.utc) - timedelta(days=SEARCH_DAYS_BACK)
    
    print(f"\n{'='*80}")
    print("正在搜索 Twitter 推文...")
    print(f"{'='*80}\n")
    print(f"查询语句: {query}")
    print(f"开始时间: {start_time.strftime('%Y-%m-%d %H:%M:%S UTC')}\n")
    
    # 执行搜索
    search_result = client.search_recent_tweets(
        query=query,
        start_time=start_time,
        max_results=MAX_RESULTS,
        sort_order=SORT_ORDER
    )
    
    if search_result and search_result.get('data'):
        tweet_count = len(search_result['data'])
        print(f"\n✓ 搜索完成")
        print(f"  - 找到推文: {tweet_count} 条")
        
        meta = search_result.get('meta', {})
        if meta:
            print(f"  - 结果数: {meta.get('result_count', 0)}")
            if 'next_token' in meta:
                print(f"  - 有更多结果可用 (next_token 存在)")
    else:
        print("\n✗ 未找到相关推文")
else:
    print("✗ 缺少 Twitter Bearer Token，无法初始化客户端")
    client = None
    search_result = None

✓ Twitter API 客户端初始化成功

正在搜索 Twitter 推文...

查询语句: (("autonomous driving") OR ("embodied intelligence") OR ("large language model") OR ("artificial intelligence") OR ("Tesla FSD") OR ("humanoid robot") OR OpenAI OR DeepSeek) lang:en -is:retweet -is:reply
开始时间: 2025-11-03 13:22:47 UTC


✓ 搜索完成
  - 找到推文: 100 条
  - 结果数: 100


## 步骤7: 展示搜索结果

显示推文内容、作者信息和互动数据

In [7]:
def display_tweets(result: Dict[str, Any], limit: int = 10):
    """
    显示推文搜索结果
    
    参数:
        result: 搜索结果字典
        limit: 显示的推文数量限制
    """
    if not result or not result.get('data'):
        print("✗ 无搜索结果")
        return
    
    tweets = result['data']
    users_dict = {}
    
    # 创建用户ID到用户信息的映射
    if 'includes' in result and 'users' in result['includes']:
        for user in result['includes']['users']:
            users_dict[user['id']] = user
    
    print("\n" + "="*80)
    print(f"{'推文搜索结果':^76}")
    print("="*80)
    
    # 显示前 N 条推文
    for i, tweet in enumerate(tweets[:limit], 1):
        tweet_id = tweet.get('id')
        text = tweet.get('text', '')
        author_id = tweet.get('author_id')
        created_at = tweet.get('created_at', '')
        metrics = tweet.get('public_metrics', {})
        
        # 获取作者信息
        author = users_dict.get(author_id, {})
        author_name = author.get('name', 'Unknown')
        author_username = author.get('username', 'unknown')
        is_verified = author.get('verified', False)
        
        # 格式化时间
        if created_at:
            try:
                dt = datetime.fromisoformat(created_at.replace('Z', '+00:00'))
                created_at_str = dt.strftime('%Y-%m-%d %H:%M:%S')
            except:
                created_at_str = created_at
        else:
            created_at_str = 'N/A'
        
        # 显示推文
        print(f"\n[{i}] {'✓' if is_verified else ''}@{author_username} - {author_name}")
        print(f"    时间: {created_at_str}")
        print(f"    {text}")
        print(f"    📊 点赞: {metrics.get('like_count', 0)} | "
              f"转推: {metrics.get('retweet_count', 0)} | "
              f"回复: {metrics.get('reply_count', 0)} | "
              f"引用: {metrics.get('quote_count', 0)}")
        print(f"    🔗 https://twitter.com/{author_username}/status/{tweet_id}")
        print("-"*80)
    
    if len(tweets) > limit:
        print(f"\n... 还有 {len(tweets) - limit} 条推文未显示 ...")

# 显示结果
if search_result:
    display_tweets(search_result, limit=10)


                                   推文搜索结果                                   

[1] @O11671Godswill - Rolland.crypto | 𝔽rAI. base.eth
    时间: 2025-11-10 07:55:43
    No matter the market, intelligent interfaces are shaping the next era of Web3.

@wardenprotocol is building the foundation where Artificial Intelligence operates safely on chain.

Through advanced cryptography and consensus mechanisms, Warden ensures every AI decision is https://t.co/JzWwi8lSMg https://t.co/EXSkdyL8Ay
    📊 点赞: 31 | 转推: 40 | 回复: 0 | 引用: 0
    🔗 https://twitter.com/O11671Godswill/status/1987791264155074882
--------------------------------------------------------------------------------

[2] @CarmieWell - SHeRLyNwiz | 𝔽rAI
    时间: 2025-11-10 07:00:27
    A new day brings new possibilities with @FractionAI_xyz 

Where artificial intelligence blend with innovation to unlock real opportunities from data. It’s more than just technology it's the future of intelligent solution built for the next generation of Web3 

## 步骤8: 保存搜索结果

将搜索结果保存为JSON文件

In [8]:
def save_search_result(result: Dict[str, Any], filename: str = None) -> Optional[str]:
    """
    保存搜索结果到JSON文件
    
    参数:
        result: 搜索结果
        filename: 文件名（可选）
    
    返回:
        保存的文件路径
    """
    if not result:
        print("✗ 无结果可保存")
        return None
    
    # 生成文件名
    if filename is None:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"tech_news_twitter_{timestamp}.json"
    
    try:
        # 保存完整的搜索结果
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        
        file_size = os.path.getsize(filename)
        print(f"\n✓ 结果已保存到: {filename}")
        print(f"  文件大小: {file_size} 字节")
        
        return filename
    except Exception as e:
        print(f"✗ 保存文件失败: {e}")
        return None

# 保存结果
if search_result:
    saved_file = save_search_result(search_result)
    if saved_file:
        print(f"\n可以使用以下代码重新加载数据:")
        print(f"with open('{saved_file}', 'r', encoding='utf-8') as f:")
        print(f"    result = json.load(f)")


✓ 结果已保存到: tech_news_twitter_20251110_212249.json
  文件大小: 280938 字节

可以使用以下代码重新加载数据:
with open('tech_news_twitter_20251110_212249.json', 'r', encoding='utf-8') as f:
    result = json.load(f)


## 步骤9: 数据分析统计

对搜索结果进行统计分析

In [9]:
def analyze_search_result(result: Dict[str, Any]):
    """
    分析推文搜索结果
    """
    if not result or not result.get('data'):
        print("无法分析：结果为空")
        return
    
    tweets = result['data']
    users_dict = {}
    
    # 创建用户映射
    if 'includes' in result and 'users' in result['includes']:
        for user in result['includes']['users']:
            users_dict[user['id']] = user
    
    print("\n" + "="*80)
    print(f"{'推文数据统计分析':^76}")
    print("="*80)
    
    # 1. 基本统计
    print(f"\n📊 基本统计:")
    print(f"   总推文数: {len(tweets)} 条")
    
    # 2. 互动数据统计
    total_likes = 0
    total_retweets = 0
    total_replies = 0
    total_quotes = 0
    
    for tweet in tweets:
        metrics = tweet.get('public_metrics', {})
        total_likes += metrics.get('like_count', 0)
        total_retweets += metrics.get('retweet_count', 0)
        total_replies += metrics.get('reply_count', 0)
        total_quotes += metrics.get('quote_count', 0)
    
    print("\n💬 互动数据:")
    print(f"   总点赞数: {total_likes:,}")
    print(f"   总转推数: {total_retweets:,}")
    print(f"   总回复数: {total_replies:,}")
    print(f"   总引用数: {total_quotes:,}")
    print(f"   平均点赞: {total_likes // len(tweets) if tweets else 0}")
    print(f"   平均转推: {total_retweets // len(tweets) if tweets else 0}")
    
    # 3. 最受欢迎的推文
    print("\n🔥 最受欢迎推文 (Top 3):")
    sorted_tweets = sorted(tweets, 
                          key=lambda t: t.get('public_metrics', {}).get('like_count', 0), 
                          reverse=True)[:3]
    
    for i, tweet in enumerate(sorted_tweets, 1):
        author_id = tweet.get('author_id')
        author = users_dict.get(author_id, {})
        author_username = author.get('username', 'unknown')
        metrics = tweet.get('public_metrics', {})
        text_preview = tweet.get('text', '')[:100]
        
        print(f"\n   [{i}] @{author_username}")
        print(f"       {text_preview}...")
        print(f"       点赞: {metrics.get('like_count', 0)} | 转推: {metrics.get('retweet_count', 0)}")
    
    # 4. 活跃用户
    author_counts = {}
    for tweet in tweets:
        author_id = tweet.get('author_id')
        author_counts[author_id] = author_counts.get(author_id, 0) + 1
    
    if author_counts:
        print("\n👤 最活跃用户 (Top 5):")
        top_authors = sorted(author_counts.items(), key=lambda x: x[1], reverse=True)[:5]
        
        for author_id, count in top_authors:
            author = users_dict.get(author_id, {})
            author_name = author.get('name', 'Unknown')
            author_username = author.get('username', 'unknown')
            print(f"     @{author_username} ({author_name}): {count} 条推文")
    
    # 5. 语言分布
    lang_counts = {}
    for tweet in tweets:
        lang = tweet.get('lang', 'unknown')
        lang_counts[lang] = lang_counts.get(lang, 0) + 1
    
    if lang_counts:
        print("\n🌐 语言分布:")
        for lang, count in sorted(lang_counts.items(), key=lambda x: x[1], reverse=True):
            print(f"     {lang}: {count} 条")
    
    # 6. 时间分布
    date_counts = {}
    for tweet in tweets:
        created_at = tweet.get('created_at', '')
        if created_at:
            try:
                dt = datetime.fromisoformat(created_at.replace('Z', '+00:00'))
                date_str = dt.strftime('%Y-%m-%d')
                date_counts[date_str] = date_counts.get(date_str, 0) + 1
            except:
                pass
    
    if date_counts:
        print("\n📅 时间分布:")
        for date, count in sorted(date_counts.items(), reverse=True):
            print(f"     {date}: {count} 条")
    
    print("\n" + "="*80)

# 执行分析
if search_result:
    analyze_search_result(search_result)


                                  推文数据统计分析                                  

📊 基本统计:
   总推文数: 100 条

💬 互动数据:
   总点赞数: 258,067
   总转推数: 37,544
   总回复数: 9,616
   总引用数: 3,403
   平均点赞: 2580
   平均转推: 375

🔥 最受欢迎推文 (Top 3):

   [1] @CultureCrave
       A Japanese organization representing Studio Ghibli, Bandai Namco, Toei Animation and more has demand...
       点赞: 72929 | 转推: 6960

   [2] @Rahll
       "...internal Slack messages and emails discussing the mass deletion of a pirated books dataset."

Op...
       点赞: 47811 | 转推: 7090

   [3] @RemmeltE
       Lawyer jumps on stage to subpoena Sam Altman. 

Sam must now join the court hearing with the activis...
       点赞: 38307 | 转推: 3075

👤 最活跃用户 (Top 5):
     @cihan121011271 (FIZYO): 3 条推文
     @itssoliu (liu (❖,❖)): 2 条推文
     @O11671Godswill (Rolland.crypto | 𝔽rAI. base.eth): 1 条推文
     @CarmieWell (SHeRLyNwiz | 𝔽rAI): 1 条推文
     @popos_ai (Jihad Prodhan): 1 条推文

🌐 语言分布:
     en: 100 条

📅 时间分布:
     2025-11-10: 13 条
     2025-11-09: 18 条

## 步骤10: 完整流程封装

将以上步骤封装成一个完整的函数

In [10]:
def acquire_latest_news_twitter(
    bearer_token: str,
    fields: List[str],
    focus_topics: List[str],
    days_back: int = 7,
    max_results: int = 100,
    sort_order: str = "relevancy",
    language: str = "en",
    exclude_retweets: bool = True,
    exclude_replies: bool = True,
    save_to_file: bool = True,
    show_analysis: bool = True
) -> Optional[Dict[str, Any]]:
    """
    完整的 Twitter 新闻获取流程
    
    参数:
        bearer_token: Twitter API Bearer Token
        fields: 技术领域列表
        focus_topics: 重点关注主题列表
        days_back: 搜索最近几天 (最多7天)
        max_results: 最大结果数 (10-100)
        sort_order: 排序方式 ('relevancy' 或 'recency')
        language: 语言过滤
        exclude_retweets: 是否排除转推
        exclude_replies: 是否排除回复
        save_to_file: 是否保存到文件
        show_analysis: 是否显示统计分析
    
    返回:
        搜索结果
    """
    print("\n" + "="*80)
    print(f"{'开始获取 Twitter 最新技术资讯':^76}")
    print("="*80)
    
    # 1. 初始化客户端
    print("\n[1/4] 初始化 Twitter API 客户端...")
    client = TwitterSearchClient(bearer_token)
    
    # 2. 构建查询
    print("\n[2/4] 构建搜索查询...")
    all_keywords = fields + focus_topics
    query = client.build_query(
        keywords=all_keywords,
        language=language,
        exclude_retweets=exclude_retweets,
        exclude_replies=exclude_replies
    )
    print(f"   查询语句: {query}")
    
    # 3. 执行搜索
    print("\n[3/4] 执行 Twitter 搜索...")
    start_time = datetime.now(timezone.utc) - timedelta(days=days_back)
    
    result = client.search_recent_tweets(
        query=query,
        start_time=start_time,
        max_results=max_results,
        sort_order=sort_order
    )
    
    if not result or not result.get('data'):
        print("\n✗ 流程终止：搜索失败或无结果")
        return None
    
    # 4. 显示结果
    print("\n[4/4] 显示搜索结果...")
    display_tweets(result, limit=10)
    
    # 5. 保存文件
    if save_to_file:
        save_search_result(result)
    
    # 6. 统计分析
    if show_analysis:
        analyze_search_result(result)
    
    print("\n" + "="*80)
    print(f"{'流程完成！':^76}")
    print("="*80)
    
    return result

print("✓ 完整流程函数已定义")
print("\n快速使用示例:")
print("""
# 获取 Twitter 最新技术资讯
result = acquire_latest_news_twitter(
    bearer_token=TWITTER_BEARER_TOKEN,
    fields=["autonomous driving", "embodied intelligence", "large language model"],
    focus_topics=["Tesla FSD", "humanoid robot", "OpenAI"],
    days_back=7,
    max_results=100,
    sort_order="relevancy",
    language="en",
    save_to_file=True,
    show_analysis=True
)
""")

✓ 完整流程函数已定义

快速使用示例:

# 获取 Twitter 最新技术资讯
result = acquire_latest_news_twitter(
    bearer_token=TWITTER_BEARER_TOKEN,
    fields=["autonomous driving", "embodied intelligence", "large language model"],
    focus_topics=["Tesla FSD", "humanoid robot", "OpenAI"],
    days_back=7,
    max_results=100,
    sort_order="relevancy",
    language="en",
    save_to_file=True,
    show_analysis=True
)



## 总结

本笔记本演示了如何使用 Twitter API v2 获取最新技术资讯，包括：

1. ✅ 安装和配置 Twitter API 客户端
2. ✅ 构建复杂的搜索查询
3. ✅ 搜索最近7天的推文
4. ✅ 获取推文、作者、媒体等扩展数据
5. ✅ 展示推文内容和互动数据
6. ✅ 保存搜索结果
7. ✅ 数据统计分析
8. ✅ 完整流程封装

### 核心优势

#### 1. 实时性强
- **最新资讯**: 获取最近7天的实时推文
- **时效性**: Twitter 是技术资讯的第一手来源
- **热度排序**: 支持按相关性或时间排序

#### 2. 数据丰富
- **完整推文**: 获取推文文本、时间、语言等
- **作者信息**: 用户名、认证状态、简介等
- **互动数据**: 点赞、转推、回复、引用数
- **媒体附件**: 图片、视频等媒体内容

#### 3. 灵活查询
- **复杂查询**: 支持 OR、AND、NOT 等逻辑运算
- **过滤选项**: 语言、转推、回复等过滤
- **时间范围**: 精确到秒的时间过滤

### 获取 Twitter API Bearer Token

1. 访问 [Twitter Developer Portal](https://developer.twitter.com/)
2. 创建一个项目 (Project) 和应用 (App)
3. 在应用的 "Keys and tokens" 页面生成 Bearer Token
4. 复制 Bearer Token 并保存到 `.env` 文件

**配置示例** (`.env` 文件):
```bash
TWITTER_BEARER_TOKEN=AAAAAAAAAAAAAAAAAAAAAA...
```

### 搜索查询语法

Twitter API 支持强大的查询语法：

```python
# 基础查询
"AI"                          # 包含 AI
"machine learning"            # 精确短语匹配

# 逻辑运算
"AI OR ML"                    # 包含 AI 或 ML
"AI -GPT"                     # 包含 AI 但不包含 GPT

# 过滤器
"lang:en"                     # 英文推文
"-is:retweet"                # 排除转推
"-is:reply"                  # 排除回复
"has:media"                   # 包含媒体
"is:verified"                 # 来自认证用户
```

### API 限制

**Twitter API v2 Recent Search (免费版)**:
- 时间范围: 最近 7 天
- 请求限制: 450 次/15分钟 (Essential 级别)
- 每次最多: 100 条推文
- 每月上限: 500,000 条推文 (Essential 级别)

**升级选项**:
- Basic ($100/月): 10,000 条/月
- Pro ($5,000/月): 1,000,000 条/月 + Full Archive Search

### 使用建议

#### 1. 优化查询
```python
# 使用具体关键词
fields = ["autonomous driving", "Tesla FSD"]

# 排除噪音
exclude_retweets=True
exclude_replies=True
```

#### 2. 控制结果数量
```python
# 相关性排序获取高质量推文
sort_order="relevancy"
max_results=100
```

#### 3. 分页获取更多结果
```python
# 使用 next_token 获取下一页
if 'next_token' in result['meta']:
    next_token = result['meta']['next_token']
    # 使用 next_token 继续搜索
```

### 与其他方案对比

| 特性 | Twitter API | 千帆AI搜索 | 腾讯混元 |
|------|------------|-----------|----------|
| 数据来源 | Twitter | 百度搜索 | 网络搜索 |
| 实时性 | 极高 | 高 | 高 |
| 时间范围 | 7天(免费) | 灵活 | 灵活 |
| AI总结 | 否 | 是 | 是 |
| 互动数据 | 完整 | 无 | 无 |
| 作者信息 | 完整 | 有限 | 有限 |
| 免费额度 | 有限制 | 100次/天 | 无 |
| 适用场景 | 社交媒体热点 | 综合资讯 | AI分析 |

### 注意事项

1. **API 限制**: 注意请求频率和月度限制
2. **时间范围**: 免费版只能搜索最近7天
3. **数据质量**: Twitter 内容质量参差不齐
4. **语言过滤**: 建议使用 `lang:en` 过滤英文内容
5. **认证用户**: 可使用 `is:verified` 过滤认证用户

### 下一步扩展

- ✅ 添加分页功能获取更多结果
- ✅ 实现情感分析
- ✅ 关键词趋势分析
- ✅ 用户影响力评估
- ✅ 媒体内容下载
- ✅ 定时监控特定话题
- ✅ 与AI模型结合生成摘要